<a href="https://colab.research.google.com/github/rohleet/darkknight/blob/main/model_training_and_evaluation/bert_phishing_email_detection_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets
!pip install nltk

In [2]:
from datasets import load_dataset

dataset = load_dataset("zefang-liu/phishing-email-dataset")
# Columns: 'text', 'label'

dataset

README.md:   0%|          | 0.00/616 [00:00<?, ?B/s]

Phishing_Email.csv: reconstructing file:   0%|          |  0.00B / 52.0MB            

Phishing_Email.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/18650 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0', 'Email Text', 'Email Type'],
        num_rows: 18650
    })
})

In [3]:
#Cleaning the dataset
MAX_CHARS = 25000
clean_dataset = dataset.filter(
    lambda x: x["Email Text"] is not None and 0 < len(x["Email Text"].strip()) <= 25000
)

Filter:   0%|          | 0/18650 [00:00<?, ? examples/s]

In [6]:
#Train test validation split

from sklearn.model_selection import train_test_split
X = clean_dataset['train']['Email Text']
y = clean_dataset['train']['Email Type']


X_train_val,X_test,y_train_val,y_test = train_test_split (
    X,
    y,
    test_size = 0.2,
    random_state = 42,
    stratify=y
)

X_train,X_val,y_train,y_val = train_test_split (
    X_train_val,
    y_train_val,
    test_size = 0.1,
    random_state = 42,
    stratify=y_train_val
)

In [7]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
# 1. Clean the text data by replacing None values with empty strings

# 2. Get the token length for each email (without truncating them)
# We use a sample of 2000-5000 rows if the dataset is massive, or the whole thing if it runs fast
# token_lengths = [len(tokenizer.encode(text, add_special_tokens=True)) for text in texts]

# # 3. Calculate percentiles using numpy
# import numpy as np

# max_len = max(token_lengths)
# p95 = np.percentile(token_lengths, 95)
# p99 = np.percentile(token_lengths, 99)

# print(f"Absolute Maximum Token Length: {max_len}")
# print(f"95% of emails are shorter than: {int(p95)} tokens")
# print(f"99% of emails are shorter than: {int(p99)} tokens")
clean_dataset = clean_dataset.remove_columns(['Unnamed: 0'])
clean_dataset = clean_dataset["train"]
clean_dataset

Dataset({
    features: ['Email Text', 'Email Type'],
    num_rows: 18564
})

In [11]:
from datasets import Dataset

train_ds = Dataset.from_dict({"Email Text": X_train, "Email Type": y_train})
val_ds   = Dataset.from_dict({"Email Text": X_val, "Email Type": y_val})
test_ds  = Dataset.from_dict({"Email Text": X_test, "Email Type": y_test})

In [20]:
label_map = {"Safe Email": 0, "Phishing Email": 1}

def tokenize_func(email):
  tokenized = tokenizer(
        email["Email Text"],
        padding="max_length",
        truncation=True,
        max_length=512,
        stride=128,
        return_overflowing_tokens=True,
    )

  sample_mapping = tokenized.pop("overflow_to_sample_mapping")
  tokenized["labels"] = [label_map[email["Email Type"][i]] for i in sample_mapping]

  return tokenized

tokenized_train_dataset = train_ds.map(tokenize_func,batched=True,remove_columns=train_ds.column_names)
tokenized_val_dataset = val_ds.map(tokenize_func,batched=True,remove_columns=val_ds.column_names)
tokenized_test_dataset = test_ds.map(tokenize_func,batched=True,remove_columns=test_ds.column_names)

Map:   0%|          | 0/13365 [00:00<?, ? examples/s]

Map:   0%|          | 0/1486 [00:00<?, ? examples/s]

Map:   0%|          | 0/3713 [00:00<?, ? examples/s]

In [14]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir = "./results",
    eval_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    save_total_limit=2,
    load_best_model_at_end=True,
    logging_steps=100,
    fp16=True ,
    save_strategy="epoch"
)

In [15]:
!pip install evaluate
from transformers import Trainer
from evaluate import load

metric = load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [21]:
from transformers import AutoModelForSequenceClassification

# Initializing BERT model
model_name = "bert-base-uncased"
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [22]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_val_dataset,
    compute_metrics=compute_metrics
)

In [23]:
trainer.train()


Epoch,Training Loss,Validation Loss,F1
1,0.087834,0.093092,0.969168
2,0.038787,0.056124,0.978274
3,0.025065,0.059038,0.979400
4,0.015571,0.058688,0.982971
5,0.014017,0.071075,0.979400


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=6415, training_loss=0.04435093446532359, metrics={'train_runtime': 2824.2307, 'train_samples_per_second': 36.325, 'train_steps_per_second': 2.271, 'total_flos': 2.69925631693824e+16, 'train_loss': 0.04435093446532359, 'epoch': 5.0})

In [24]:
test_results = trainer.evaluate(tokenized_test_dataset)
print(test_results)

Training Loss,Validation Loss,Epoch,F1
0.014017,0.117942,5,0.968952


{'eval_loss': 0.11794205754995346, 'eval_f1': 0.9689521345407504}


In [25]:
save_path = "/content/drive/MyDrive/my_phishing_bert_model"

# Saves the best loaded model weights and config
trainer.save_model(save_path)

# Saves the tokenizer vocabulary alongside the model so you can reload both together
tokenizer.save_pretrained(save_path)
print("Saved successfully to:", save_path)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved successfully to: /content/drive/MyDrive/my_phishing_bert_model


In [26]:
from sklearn.metrics import classification_report, confusion_matrix

preds_output = trainer.predict(tokenized_test_dataset)
preds = preds_output.predictions.argmax(axis=-1)
labels = preds_output.label_ids

print(confusion_matrix(labels, preds))
print(classification_report(labels, preds, target_names=["Safe Email", "Phishing Email"]))


[[3404  105]
 [  39 2247]]
                precision    recall  f1-score   support

    Safe Email       0.99      0.97      0.98      3509
Phishing Email       0.96      0.98      0.97      2286

      accuracy                           0.98      5795
     macro avg       0.97      0.98      0.97      5795
  weighted avg       0.98      0.98      0.98      5795



In [2]:
from datasets import load_dataset

new_test_set = load_dataset("SetFit/enron_spam",split='train')

new_test_set

README.md:   0%|          | 0.00/176 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


train.jsonl: reconstructing file:   0%|          |  0.00B /  101MB            

train.jsonl: downloading bytes:           |  0.00B            

test.jsonl: reconstructing file:   0%|          |  0.00B / 6.27MB            

test.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/31716 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Dataset({
    features: ['message_id', 'text', 'label', 'label_text', 'subject', 'message', 'date'],
    num_rows: 31716
})

In [25]:
from datasets import Dataset

MAX_CHARS = 25000
new_clean = new_test_set.filter(
    lambda x: x["text"] is not None and 0 < len(x["text"].strip()) <= 25000
)

dataset = load_dataset("zefang-liu/phishing-email-dataset",split='train')
clean_dataset = dataset.filter(
    lambda x: x["Email Text"] is not None and 0 < len(x["Email Text"].strip()) <= 25000
)

original_texts = set(clean_dataset["Email Text"])

truly_unseen_set = new_clean.filter(lambda x: x["text"] not in original_texts)
print(f"Truly unseen emails remaining: {len(truly_unseen_set)} out of {len(new_clean)}")


X = truly_unseen_set['text']
y = truly_unseen_set['label']

test_di = Dataset.from_dict({'text' : X, 'label' : y})

Truly unseen emails remaining: 31013 out of 31570


In [26]:
label_map = {0: 0, 1: 1}

def tokenize_test(email):
    tokenized = tokenizer(
        email["text"],
        padding="max_length",
        truncation=True,
        max_length=512,
        stride=128,
        return_overflowing_tokens=True,
    )

    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    tokenized["labels"] = [label_map[email["label"][i]] for i in sample_mapping]

    return tokenized


tokenized_test = test_di.map(
    tokenize_test,
    batched=True,
    remove_columns=test_di.column_names
)

Map:   0%|          | 0/31013 [00:00<?, ? examples/s]

In [27]:
print(tokenized_test)

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 41228
})


In [12]:
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification

model_path = "/content/drive/MyDrive/my_phishing_bert_model"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
model.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,),

In [28]:
from transformers import Trainer, TrainingArguments
from sklearn.metrics import classification_report, confusion_matrix

eval_args = TrainingArguments(
    output_dir="./eval_temp",
    per_device_eval_batch_size=32,
    fp16=True,
)

trainer = Trainer(
    model=model,
    args=eval_args,
)

preds_output = trainer.predict(tokenized_test)

preds = preds_output.predictions.argmax(axis=-1)
labels = preds_output.label_ids

print(confusion_matrix(labels, preds))
print(classification_report(labels, preds, target_names=["Safe Email", "Phishing Email"]))

[[19995   122]
 [  258 20853]]
                precision    recall  f1-score   support

    Safe Email       0.99      0.99      0.99     20117
Phishing Email       0.99      0.99      0.99     21111

      accuracy                           0.99     41228
     macro avg       0.99      0.99      0.99     41228
  weighted avg       0.99      0.99      0.99     41228

